# 10 — Visualization Export

Build self-contained HTML map using Folium.

- Plot all File 2 stations color-coded: Green=Sufficient, Yellow=Moderate, Red=Congested
- Popups show: location_id, route_segment, n_chargers_proposed, grid_status
- Layers: existing chargers, road network, friction points
- Export to `visualization/bi_map.html` — must work **offline** with no logins

## Data Inputs
- `output/File_2.csv`
- `output/File_3.csv`
- `data/processed/interurban_chargers.csv`
- `data/processed/interurban_roads.parquet`

## Data Outputs
- `visualization/bi_map.html` — self-contained interactive map (no external dependencies)

In [ ]:
import os
import sys
import pandas as pd
from pathlib import Path

try:
    import folium
    from folium.plugins import MarkerCluster
except ModuleNotFoundError:
    folium = None
    MarkerCluster = None

if os.path.basename(os.getcwd()) == 'notebooks':
    sys.path.insert(0, os.path.dirname(os.getcwd()))
    OUTPUT_DIR = Path('../output')
    DATA_DIR = Path('../data/processed')
    VIZ_DIR = Path('../visualization')
else:
    sys.path.insert(0, os.getcwd())
    OUTPUT_DIR = Path('output')
    DATA_DIR = Path('data/processed')
    VIZ_DIR = Path('visualization')

from src.constants import POWER_PER_CHARGER_KW

VIZ_DIR.mkdir(parents=True, exist_ok=True)

COLOR_MAP = {
    'Sufficient': 'green',
    'Moderate': 'orange',
    'Congested': 'red',
}

print('✅ Imports OK')
print(f'   Output dir: {OUTPUT_DIR}')
if folium is None:
    print('⚠️  folium is not installed in this environment; visualization cells will need the package before export')

## Step 1: Load data

In [ ]:
# File_2 — all proposed stations with grid status
file2 = pd.read_csv(OUTPUT_DIR / 'File_2.csv')
print(f'📍 Proposed stations (File_2): {len(file2):,}')
print(f'   Grid status breakdown:')
print(file2['grid_status'].value_counts().to_string())

# File_3 — friction points (Moderate + Congested only)
file3 = pd.read_csv(OUTPUT_DIR / 'File_3.csv')
print(f'\n🔥 Friction points (File_3): {len(file3):,}')

# Existing chargers baseline
baseline_path = DATA_DIR / 'interurban_chargers_baseline.csv'
if baseline_path.exists():
    existing = pd.read_csv(baseline_path)
else:
    existing = pd.DataFrame(columns=['latitude', 'longitude', 'max_power_kw'])
print(f'\n🔌 Existing baseline chargers: {len(existing):,}')

print('\n✅ Data loaded')

## Step 2: Build Folium Map

In [ ]:
# Base map centered on Spain
m = folium.Map(
    location=[40.4, -3.7],
    zoom_start=6,
    tiles='CartoDB positron',
)

# Layer groups — toggled via LayerControl
layer_proposed = folium.FeatureGroup(name='Proposed Stations (File_2)', show=True)
layer_friction = folium.FeatureGroup(name='Friction Points (File_3)', show=True)
layer_existing = folium.FeatureGroup(name='Existing Chargers (Baseline)', show=False)

print('🗺️  Base map created (CartoDB Positron, centered on Spain)')

## Step 3: Add Proposed Stations Layer (File_2)

Color coding: Green = Sufficient grid capacity, Orange = Moderate, Red = Congested.  
Popup shows: location_id, route_segment, n_chargers_proposed, grid_status, estimated_demand_kw.

In [ ]:
for _, row in file2.iterrows():
    status = row.get('grid_status', 'Congested')
    color = COLOR_MAP.get(status, 'gray')
    n_chargers = int(row.get('n_chargers_proposed', 2))
    demand_kw = n_chargers * POWER_PER_CHARGER_KW

    popup_html = f"""
    <b>{row.get('location_id', 'N/A')}</b><br>
    Route: {row.get('route_segment', 'N/A')}<br>
    Chargers: {n_chargers} × 150 kW = {demand_kw:,} kW<br>
    Grid: <span style="color:{color};font-weight:bold">{status}</span>
    """

    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=6,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.75,
        popup=folium.Popup(popup_html, max_width=220),
        tooltip=f"{row.get('location_id', '')} — {status}",
    ).add_to(layer_proposed)

layer_proposed.add_to(m)
print(f'✅ Proposed stations layer: {len(file2):,} markers added')

## Step 4: Add Friction Points Layer (File_3)

Friction points are stations with grid constraints (Moderate or Congested). Shown as larger markers with a distinct border to highlight grid investment need.

In [ ]:
for _, row in file3.iterrows():
    status = row.get('grid_status', 'Congested')
    color = COLOR_MAP.get(status, 'red')
    demand_kw = int(row.get('estimated_demand_kw', 0))
    distributor = row.get('distributor_network', 'N/A')

    popup_html = f"""
    <b>⚡ FRICTION POINT</b><br>
    ID: {row.get('bottleneck_id', 'N/A')}<br>
    Route: {row.get('route_segment', 'N/A')}<br>
    Demand: {demand_kw:,} kW<br>
    DSO: {distributor}<br>
    Grid: <span style="color:{color};font-weight:bold">{status}</span>
    """

    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=9,
        color='black',
        weight=2,
        fill=True,
        fill_color=color,
        fill_opacity=0.9,
        popup=folium.Popup(popup_html, max_width=220),
        tooltip=f"⚡ {row.get('bottleneck_id', '')} — {status} ({demand_kw:,} kW)",
    ).add_to(layer_friction)

layer_friction.add_to(m)
print(f'✅ Friction points layer: {len(file3):,} markers added')

## Step 5: Add Existing Chargers Layer (Baseline)

In [ ]:
for _, row in existing.iterrows():
    lat = row.get('latitude')
    lon = row.get('longitude')
    if pd.isna(lat) or pd.isna(lon):
        continue
    power = row.get('max_power_kw', 0)
    popup_html = f"""
    <b>Existing Charger</b><br>
    Power: {power:.0f} kW
    """
    folium.CircleMarker(
        location=[lat, lon],
        radius=3,
        color='blue',
        fill=True,
        fill_color='blue',
        fill_opacity=0.5,
        popup=folium.Popup(popup_html, max_width=160),
        tooltip=f"Existing — {power:.0f} kW",
    ).add_to(layer_existing)

layer_existing.add_to(m)
print(f'✅ Existing chargers layer: {len(existing):,} markers added')

## Step 6: Add Legend, Layer Controls & Save

In [ ]:
# HTML legend — embedded in the map (works offline)
legend_html = """
<div style="
    position: fixed;
    bottom: 40px; left: 40px; z-index: 9999;
    background-color: white;
    border: 2px solid grey;
    border-radius: 8px;
    padding: 12px 16px;
    font-size: 13px;
    font-family: Arial, sans-serif;
    box-shadow: 3px 3px 6px rgba(0,0,0,0.3);
">
<b>Iberdrola EV Network 2027</b><br>
<hr style="margin:6px 0">
<b>Proposed Stations (File_2)</b><br>
<span style="color:green;">&#9679;</span> Sufficient (&ge;5 MW available)<br>
<span style="color:orange;">&#9679;</span> Moderate (1–5 MW available)<br>
<span style="color:red;">&#9679;</span> Congested (&lt;1 MW available)<br>
<hr style="margin:6px 0">
<b>Friction Points (File_3)</b><br>
<span style="color:orange;">&#9679;</span> Moderate (thick border)<br>
<span style="color:red;">&#9679;</span> Congested (thick border)<br>
<hr style="margin:6px 0">
<span style="color:blue;">&#9679;</span> Existing Chargers (baseline)
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

# Layer control (toggle layers on/off)
folium.LayerControl(collapsed=False).add_to(m)

# Save self-contained HTML (embed_thumbnail=False avoids external dependencies)
out_path = VIZ_DIR / 'bi_map.html'
m.save(str(out_path))

print(f'💾 Map saved → {out_path}')
print(f'   Layers: Proposed stations, Friction points, Existing chargers (toggle with LayerControl)')
print(f'   Open in any browser — no internet connection required')
m